# 1. Imports & Setup

In [3]:
import re
import time
import math
import schedule
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
from collections import deque, Counter
from datetime import datetime

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from pymongo import MongoClient

# --- NEW SELENIUM IMPORTS ---
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

STOPWORDS = set(stopwords.words("english"))
stemmer = PorterStemmer()

# 2. MongoDB Connection

In [4]:
# Connect to the local MongoDB instance[cite: 1]
MONGO_URI = "mongodb://localhost:27017/"
client = MongoClient(MONGO_URI)
db = client["vertical_search_engine"]

raw_pages = db["raw_pages"]
doc_vectors = db["doc_vectors"]
term_index = db["term_index"]
crawl_log = db["crawl_log"]

raw_pages.create_index("url", unique=True)
doc_vectors.create_index("url", unique=True)
term_index.create_index("term", unique=True)

print("Connected:", db.name, "| collections:", db.list_collection_names())

Connected: vertical_search_engine | collections: ['raw_pages', 'doc_vectors', 'crawl_log', 'term_index']


# 3. Crawler

In [ ]:
SEED_URL = "https://softwarica.edu.np/courses"
ALLOWED_DOMAIN = "softwarica.edu.np"
CRAWL_DELAY_SECONDS = 2
MAX_PAGES = 100
USER_AGENT = "SoftwaricaVerticalSearchBot/1.0 (+educational IR project)"

def get_robot_parser(base_url):
    parsed = urlparse(base_url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = RobotFileParser()
    try:
        rp.set_url(robots_url)
        rp.read()
    except Exception:
        rp = None
    return rp

def can_fetch(rp, url):
    if rp is None:
        return True
    return rp.can_fetch(USER_AGENT, url)

def extract_content(html, base_url):
    """Safely extracts links and fully rendered text."""
    soup = BeautifulSoup(html, "html.parser")

    # 1. EXTRACT LINKS FIRST
    links = set()
    for a in soup.find_all("a", href=True):
        link = urljoin(base_url, a["href"]).split("#")[0]
        parsed = urlparse(link)
        if parsed.netloc.endswith(ALLOWED_DOMAIN) and parsed.scheme in ("http", "https"):
            links.add(link)

    # 2. DECOMPOSE UNWANTED TAGS
    for tag in soup(["script", "style", "nav", "footer", "header", "noscript"]):
        tag.decompose()

    title = soup.title.string.strip() if (soup.title and soup.title.string) else ""
    text = soup.get_text(separator=" ")
    text = re.sub(r"\s+", " ", text).strip()

    return title, text, links

def setup_headless_driver():
    """Configures an invisible Chrome browser for crawling."""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument(f"user-agent={USER_AGENT}")

    driver = webdriver.Chrome(options=options)
    return driver

def crawl(seed_url=SEED_URL, max_pages=MAX_PAGES):
    rp = get_robot_parser(seed_url)
    visited = set()
    queue = deque([seed_url])
    crawled_count = 0

    print("Starting Headless Chrome...")
    driver = setup_headless_driver()

    try:
        while queue and crawled_count < max_pages:
            url = queue.popleft()
            if url in visited:
                continue
            visited.add(url)

            if not can_fetch(rp, url):
                print(f"Blocked by robots.txt: {url}")
                continue

            try:
                driver.get(url)
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.TAG_NAME, "body"))
                )
                time.sleep(1) # React hydration delay
                html = driver.page_source

            except Exception as e:
                print(f"Failed to fetch {url}: {e}")
                continue

            title, text, links = extract_content(html, url)

            raw_pages.update_one(
                {"url": url},
                {"$set": {"url": url, "title": title, "text": text, "crawled_at": datetime.utcnow()}},
                upsert=True,
            )

            crawled_count += 1
            print(f"[{crawled_count}] Crawled: {url}")

            for link in links:
                if link not in visited:
                    queue.append(link)

            time.sleep(CRAWL_DELAY_SECONDS)

    finally:
        driver.quit()
        print("Closed Headless Chrome.")

    crawl_log.insert_one({"run_at": datetime.utcnow(), "pages_crawled": crawled_count})
    print(f"Crawl finished. {crawled_count} pages stored in MongoDB.")
    return crawled_count

# 4. Weekly Re-crawl Schedule

In [6]:
# Re-runs the crawl and re-indexing once a week
def scheduled_job():
    print(f"Running scheduled weekly crawl at {datetime.now()}")
    crawl()
    build_index()

schedule.every(7).days.do(scheduled_job)

def run_scheduler_blocking():
    """Run this in a separate long-lived process, not inside the notebook kernel."""
    while True:
        schedule.run_pending()
        time.sleep(60)

# 5. Text Preprocessing

In [7]:
# Lowercase, strip non-alphanumerics, tokenize, drop stopwords, and stem
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(t) for t in tokens if t not in STOPWORDS and len(t) > 2]
    return tokens

# 6. Indexing — Vector Space Model (TF-IDF)

In [8]:
# Builds the TF-IDF weighted document vectors and the inverted index for terms
def build_index():
    docs = list(raw_pages.find({}))
    N = len(docs)
    if N == 0:
        print("No documents to index. Run crawl() first.")
        return

    doc_tokens = {}
    df = Counter()

    for doc in docs:
        tokens = preprocess(doc["text"])
        doc_tokens[doc["url"]] = tokens
        for term in set(tokens):
            df[term] += 1

    idf = {term: math.log(N / (1 + freq)) + 1 for term, freq in df.items()}

    term_index.delete_many({})
    if idf:
        term_index.insert_many([{"term": t, "idf": v} for t, v in idf.items()])

    doc_vectors.delete_many({})
    for doc in docs:
        url = doc["url"]
        tokens = doc_tokens[url]
        total_terms = len(tokens) or 1
        tf = Counter(tokens)

        vector = {term: (count / total_terms) * idf.get(term, 0) for term, count in tf.items()}
        norm = math.sqrt(sum(w * w for w in vector.values())) or 1.0
        vector = {t: w / norm for t, w in vector.items()}

        doc_vectors.update_one(
            {"url": url},
            {"$set": {"url": url, "title": doc.get("title", ""), "vector": vector,
                       "indexed_at": datetime.utcnow()}},
            upsert=True,
        )

    print(f"Indexed {N} documents. Vocabulary size: {len(idf)} terms.")

# 7. Query Processing & Cosine Similarity Ranking

In [9]:
# Processes user queries and calculates cosine similarity
def build_query_vector(query):
    tokens = preprocess(query)
    tf = Counter(tokens)
    total_terms = len(tokens) or 1

    idf_docs = term_index.find({"term": {"$in": list(tf.keys())}})
    idf_map = {d["term"]: d["idf"] for d in idf_docs}

    vector = {term: (count / total_terms) * idf_map[term] for term, count in tf.items() if term in idf_map}
    norm = math.sqrt(sum(w * w for w in vector.values())) or 1.0
    return {t: w / norm for t, w in vector.items()}


def cosine_similarity(vec1, vec2):
    common_terms = set(vec1.keys()) & set(vec2.keys())
    return sum(vec1[t] * vec2[t] for t in common_terms)


def search(query, top_k=10):
    q_vector = build_query_vector(query)
    if not q_vector:
        print("No matching terms found in the index.")
        return []

    results = []
    for doc in doc_vectors.find({}):
        score = cosine_similarity(q_vector, doc["vector"])
        if score > 0:
            results.append((score, doc["url"], doc.get("title", "")))

    results.sort(key=lambda x: x[0], reverse=True)
    return results[:top_k]

# 8. Search Interface (plain notebook I/O — no frontend)

In [69]:
# Simple command-line loop for queries
def run_search_interface():
    print("Softwarica Courses — Vertical Search Engine")
    print("Type 'exit' to quit.\n")
    while True:
        query = input("Search query: ").strip()
        if query.lower() == "exit":
            break
        print(f"\nSearching for: \"{query}\"\n")
        results = search(query, top_k= 3)
        if not results:
            print("No results found.\n")
            continue
        for rank, (score, url, title) in enumerate(results, start=1):
            print(f"{rank}. [{score:.4f}] {title}\n   {url}\n")

# 9. Run the Pipeline

In [9]:
# One-time (or weekly) pipeline run
crawl()
build_index()

Starting Headless Chrome...


C:\Users\lenovo\AppData\Local\Temp\ipykernel_11432\4195846206.py:93: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  {"$set": {"url": url, "title": title, "text": text, "crawled_at": datetime.utcnow()}},


[1] Crawled: https://softwarica.edu.np/courses
[2] Crawled: https://softwarica.edu.np/student-center/news
[3] Crawled: https://softwarica.edu.np/apply
[4] Crawled: https://softwarica.edu.np/about-us/chairman-message
[5] Crawled: https://ftp.softwarica.edu.np/uploads/courses/brochure-1765166531384-448086387.pdf
[6] Crawled: https://ftp.softwarica.edu.np/uploads/courses/brochure-1764816426424-352425346.pdf
[7] Crawled: https://softwarica.edu.np/about-us/about-college
[8] Crawled: https://softwarica.edu.np/student-center/clubs/multimedia-club-of-softwarica
[9] Crawled: https://softwarica.edu.np/courses/bsc-hons-software-engineering
[10] Crawled: https://softwarica.edu.np/about-us/extended-education
[11] Crawled: https://softwarica.edu.np/student-center/notices
[12] Crawled: https://softwarica.edu.np/student-center/clubs/events-sports-club-of-softwarica
[13] Crawled: https://softwarica.edu.np/about-us/coventry-university
[14] Crawled: https://softwarica.edu.np/about-us/faq
[15] Crawled: ht

C:\Users\lenovo\AppData\Local\Temp\ipykernel_11432\4195846206.py:110: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  crawl_log.insert_one({"run_at": datetime.utcnow(), "pages_crawled": crawled_count})


Indexed 100 documents. Vocabulary size: 2527 terms.


C:\Users\lenovo\AppData\Local\Temp\ipykernel_11432\2944057987.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "indexed_at": datetime.utcnow()}},


In [10]:
# Try a query directly
for score, url, title in search("bachelor computer science"):
    print(f"{score:.4f}  {title}  ({url})")

0.3330  Programs & Courses | Softwarica College  (https://softwarica.edu.np/courses)
0.3147  BSc (Hons) Computer Science with Artificial Intelligence - Softwarica College  (https://softwarica.edu.np/courses/bsc-hons-computer-science-with-artificial-intelligence)
0.3147  BSc (Hons) Computer Science with Artificial Intelligence - Softwarica College  (https://softwarica.edu.np/course/computer-science-with-artificial-intelligence)
0.2513  BSc (Hons) Computer Science with Artificial Intelligence- What You'll Learn - Career & Scope | Softwarica College Blog  (https://softwarica.edu.np/student-center/blogs/bsc-hons-computer-science-with-artificial-intelligence-what-you-ll-learn-career-scope)
0.2448  BSc (Hons) Computer Science with Artificial Intelligence vs BCA in Nepal | Softwarica College Blog  (https://softwarica.edu.np/student-center/blogs/bsc-hons-computer-science-with-artificial-intelligence-vs-bca-in-nepal)
0.2243  Computer Science Course in Nepal- Skills, Career Scope & Opportunities

In [ ]:
# Or use the interactive loop
run_search_interface()

# Part B - Hands on Modifications

Changing seed, scope and max pages

In [37]:
SEED_URL = "https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/"
ALLOWED_DOMAIN = "pureportal.coventry.ac.uk"
MAX_PAGES = 5

In [32]:
def get_robot_parser(base_url):
    parsed = urlparse(base_url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = RobotFileParser()
    rp.set_url(robots_url)
    try:
        req = urllib.request.Request(robots_url, headers={"User-Agent": USER_AGENT})
        with urllib.request.urlopen(req, timeout=10) as resp:
            raw = resp.read().decode("utf-8", errors="ignore")
        rp.parse(raw.splitlines())   # feed rules in manually instead of rp.read()
    except Exception:
        rp = None
    return rp

In [38]:
# Rerun with new seed URL and scope
crawl(seed_url=SEED_URL, max_pages= MAX_PAGES)
build_index()

Starting Headless Chrome...


C:\Users\lenovo\AppData\Local\Temp\ipykernel_2160\2412393138.py:93: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  {"$set": {"url": url, "title": title, "text": text, "crawled_at": datetime.utcnow()}},


[1] Crawled: https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/
[2] Crawled: https://pureportal.coventry.ac.uk/en/projects/developing-research-engagement-for-happy-healthy-lives/
[3] Crawled: https://pureportal.coventry.ac.uk/en/prizes/poster-2nd-place-in-research-that-transforms-and-improves-practic/
[4] Crawled: https://pureportal.coventry.ac.uk/en/organisations/
[5] Crawled: https://pureportal.coventry.ac.uk/en/organisations/research-division/
Closed Headless Chrome.
Crawl finished. 5 pages stored in MongoDB.


C:\Users\lenovo\AppData\Local\Temp\ipykernel_2160\2412393138.py:110: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  crawl_log.insert_one({"run_at": datetime.utcnow(), "pages_crawled": crawled_count})
C:\Users\lenovo\AppData\Local\Temp\ipykernel_2160\2944057987.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "indexed_at": datetime.utcnow()}},


Indexed 200 documents. Vocabulary size: 4792 terms.


Adding stop word: Softwarica

In [50]:
print("Before adding Stopword: Softwarica")
search("softwarica courses")


Before adding Stopword: Softwarica


[(0.37544821096435815,
  'https://softwarica.edu.np/student-center/blogs/is-a-3-years-it-course-in-nepal-worth-it-find-out-now',
  'Is a 3 Years IT Course in Nepal Worth It? Find Out Now | Softwarica College Blog'),
 (0.32125170331651576,
  'https://softwarica.edu.np/student-center/blogs/why-choose-softwarica-college-for-it-courses-in-nepal',
  'Why Choose Softwarica College for IT Courses in Nepal | Softwarica College Blog'),
 (0.280650406442601,
  'https://softwarica.edu.np/student-center/blogs/softwarica-college-admission-fees-eligibility-scholarships',
  'Softwarica College Admission - Fees - Eligibility &  Scholarships | Softwarica College Blog'),
 (0.27040720840959165,
  'https://softwarica.edu.np/student-center/blogs/the-best-it-courses-in-nepal-which-one-should-you-choose-in-2026-',
  'The Best IT Courses in Nepal: Which One Should You Choose in 2026? | Softwarica College Blog'),
 (0.26880247449825806,
  'https://softwarica.edu.np/student-center/blogs/ai-course-in-nepal-high-de

In [56]:
print("After adding Stopword: Softwarica")
STOPWORDS.add("softwarica")
build_index()
search("softwarica courses")

After adding Stopword: Softwarica


C:\Users\lenovo\AppData\Local\Temp\ipykernel_2160\2944057987.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "indexed_at": datetime.utcnow()}},


Indexed 200 documents. Vocabulary size: 4791 terms.


[(0.37804336076220096,
  'https://softwarica.edu.np/student-center/blogs/is-a-3-years-it-course-in-nepal-worth-it-find-out-now',
  'Is a 3 Years IT Course in Nepal Worth It? Find Out Now | Softwarica College Blog'),
 (0.2692736967387446,
  'https://softwarica.edu.np/student-center/blogs/the-best-it-courses-in-nepal-which-one-should-you-choose-in-2026-',
  'The Best IT Courses in Nepal: Which One Should You Choose in 2026? | Softwarica College Blog'),
 (0.2497196075014389,
  'https://softwarica.edu.np/student-center/blogs/why-you-should-study-ai-now-before-it-s-too-late-',
  'Why You Should Study AI Now (Before It’s Too Late) | Softwarica College Blog'),
 (0.24556181415356673,
  'https://softwarica.edu.np/courses',
  'Programs & Courses | Softwarica College'),
 (0.24511154635586505,
  'https://softwarica.edu.np/courses/bsc-hons-ethical-hacking-and-cybersecurity',
  'BSc (Hons) Ethical Hacking and Cybersecurity - Softwarica College'),
 (0.23470911417481488,
  'https://softwarica.edu.np/c

#### Breaking normalization: removing normalization step

In [57]:
# Commenting out L2 normalization step
def build_index():
    docs = list(raw_pages.find({}))
    N = len(docs)
    if N == 0:
        print("No documents to index. Run crawl() first.")
        return

    doc_tokens = {}
    df = Counter()

    for doc in docs:
        tokens = preprocess(doc["text"])
        doc_tokens[doc["url"]] = tokens
        for term in set(tokens):
            df[term] += 1

    idf = {term: math.log(N / (1 + freq)) + 1 for term, freq in df.items()}

    term_index.delete_many({})
    if idf:
        term_index.insert_many([{"term": t, "idf": v} for t, v in idf.items()])

    doc_vectors.delete_many({})
    for doc in docs:
        url = doc["url"]
        tokens = doc_tokens[url]
        total_terms = len(tokens) or 1
        tf = Counter(tokens)

        vector = {term: (count / total_terms) * idf.get(term, 0) for term, count in tf.items()}
        # norm = math.sqrt(sum(w * w for w in vector.values())) or 1.0
        # vector = {t: w / norm for t, w in vector.items()}

        doc_vectors.update_one(
            {"url": url},
            {"$set": {"url": url, "title": doc.get("title", ""), "vector": vector,
                       "indexed_at": datetime.utcnow()}},
            upsert=True,
        )

    print(f"Indexed {N} documents. Vocabulary size: {len(idf)} terms.")

In [58]:
build_index()
search("softwarica courses")

C:\Users\lenovo\AppData\Local\Temp\ipykernel_2160\21748288.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "indexed_at": datetime.utcnow()}},


Indexed 200 documents. Vocabulary size: 4791 terms.


[(0.08725758323302157,
  'https://softwarica.edu.np/courses',
  'Programs & Courses | Softwarica College'),
 (0.07578649769721815,
  'https://softwarica.edu.np/student-center/blogs/is-a-3-years-it-course-in-nepal-worth-it-find-out-now',
  'Is a 3 Years IT Course in Nepal Worth It? Find Out Now | Softwarica College Blog'),
 (0.07206103783850658,
  'https://softwarica.edu.np/courses/bsc-hons-ethical-hacking-and-cybersecurity',
  'BSc (Hons) Ethical Hacking and Cybersecurity - Softwarica College'),
 (0.067867009181239,
  'https://softwarica.edu.np/about-us/faq',
  'FAQ - About Us | Softwarica College'),
 (0.06486404417321957,
  'https://softwarica.edu.np/courses/msc-data-science-and-computational-intelligence',
  'MSc Data Science and Computational Intelligence - Softwarica College'),
 (0.0610803082631151,
  'https://softwarica.edu.np/courses/bsc-hons-computer-science-with-artificial-intelligence',
  'BSc (Hons) Computer Science with Artificial Intelligence - Softwarica College'),
 (0.061

#### Logging every crawl

In [62]:
# log every crawl
for entry in crawl_log.find():
    print(entry)

{'_id': ObjectId('6a76d7873c73b53ec142294c'), 'run_at': datetime.datetime(2026, 8, 8, 7, 15, 19, 469000), 'pages_crawled': 100}
{'_id': ObjectId('6a77520eced0699aec415c35'), 'run_at': datetime.datetime(2026, 8, 8, 15, 58, 6, 145000), 'pages_crawled': 1}
{'_id': ObjectId('6a7752fcced0699aec416615'), 'run_at': datetime.datetime(2026, 8, 8, 16, 2, 4, 542000), 'pages_crawled': 1}
{'_id': ObjectId('6a77535cced0699aec416ff5'), 'run_at': datetime.datetime(2026, 8, 8, 16, 3, 40, 212000), 'pages_crawled': 0}
{'_id': ObjectId('6a775381ced0699aec4179d5'), 'run_at': datetime.datetime(2026, 8, 8, 16, 4, 17, 384000), 'pages_crawled': 0}
{'_id': ObjectId('6a7753faced0699aec4183b5'), 'run_at': datetime.datetime(2026, 8, 8, 16, 6, 18, 200000), 'pages_crawled': 0}
{'_id': ObjectId('6a775416ced0699aec418d95'), 'run_at': datetime.datetime(2026, 8, 8, 16, 6, 46, 514000), 'pages_crawled': 0}
{'_id': ObjectId('6a775483ced0699aec419775'), 'run_at': datetime.datetime(2026, 8, 8, 16, 8, 35, 992000), 'pages_craw

Adding skipped_by_robots counter

In [63]:
# adding skipped_by_robots counter
def crawl(seed_url=SEED_URL, max_pages=MAX_PAGES):
    rp = get_robot_parser(seed_url)
    visited = set()
    queue = deque([seed_url])
    crawled_count = 0
    skipped_by_robots = 0

    print("Starting Headless Chrome...")
    driver = setup_headless_driver()

    try:
        while queue and crawled_count < max_pages:
            url = queue.popleft()
            if url in visited:
                continue
            visited.add(url)

            if not can_fetch(rp, url):
                print(f"Blocked by robots.txt: {url}")
                skipped_by_robots += 1
                continue

            try:
                driver.get(url)
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.TAG_NAME, "body"))
                )
                time.sleep(1) # React hydration delay
                html = driver.page_source

            except Exception as e:
                print(f"Failed to fetch {url}: {e}")
                continue

            title, text, links = extract_content(html, url)

            raw_pages.update_one(
                {"url": url},
                {"$set": {"url": url, "title": title, "text": text, "crawled_at": datetime.utcnow()}},
                upsert=True,
            )

            crawled_count += 1
            print(f"[{crawled_count}] Crawled: {url}")

            for link in links:
                if link not in visited:
                    queue.append(link)

            time.sleep(CRAWL_DELAY_SECONDS)

    finally:
        driver.quit()
        print("Closed Headless Chrome.")

    crawl_log.insert_one({
        "run_at": datetime.utcnow(),
        "pages_crawled": crawled_count,
        "skipped_by_robots": skipped_by_robots
    })
    print(f"Crawl finished. {crawled_count} pages stored, {skipped_by_robots} skipped by robots.txt.")
    return crawled_count

In [64]:
crawl(seed_url=SEED_URL, max_pages= MAX_PAGES)
build_index()

Starting Headless Chrome...


C:\Users\lenovo\AppData\Local\Temp\ipykernel_2160\2131405999.py:40: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  {"$set": {"url": url, "title": title, "text": text, "crawled_at": datetime.utcnow()}},


[1] Crawled: https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/
[2] Crawled: https://pureportal.coventry.ac.uk/en/projects/developing-research-engagement-for-happy-healthy-lives/
[3] Crawled: https://pureportal.coventry.ac.uk/en/prizes/poster-2nd-place-in-research-that-transforms-and-improves-practic/
[4] Crawled: https://pureportal.coventry.ac.uk/en/organisations/
[5] Crawled: https://pureportal.coventry.ac.uk/en/organisations/research-division/
Closed Headless Chrome.
Crawl finished. 5 pages stored, 0 skipped by robots.txt.


C:\Users\lenovo\AppData\Local\Temp\ipykernel_2160\2131405999.py:58: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "run_at": datetime.utcnow(),
C:\Users\lenovo\AppData\Local\Temp\ipykernel_2160\21748288.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "indexed_at": datetime.utcnow()}},


Indexed 200 documents. Vocabulary size: 4791 terms.


In [65]:
# log every crawl
for entry in crawl_log.find():
    print(entry)

{'_id': ObjectId('6a76d7873c73b53ec142294c'), 'run_at': datetime.datetime(2026, 8, 8, 7, 15, 19, 469000), 'pages_crawled': 100}
{'_id': ObjectId('6a77520eced0699aec415c35'), 'run_at': datetime.datetime(2026, 8, 8, 15, 58, 6, 145000), 'pages_crawled': 1}
{'_id': ObjectId('6a7752fcced0699aec416615'), 'run_at': datetime.datetime(2026, 8, 8, 16, 2, 4, 542000), 'pages_crawled': 1}
{'_id': ObjectId('6a77535cced0699aec416ff5'), 'run_at': datetime.datetime(2026, 8, 8, 16, 3, 40, 212000), 'pages_crawled': 0}
{'_id': ObjectId('6a775381ced0699aec4179d5'), 'run_at': datetime.datetime(2026, 8, 8, 16, 4, 17, 384000), 'pages_crawled': 0}
{'_id': ObjectId('6a7753faced0699aec4183b5'), 'run_at': datetime.datetime(2026, 8, 8, 16, 6, 18, 200000), 'pages_crawled': 0}
{'_id': ObjectId('6a775416ced0699aec418d95'), 'run_at': datetime.datetime(2026, 8, 8, 16, 6, 46, 514000), 'pages_crawled': 0}
{'_id': ObjectId('6a775483ced0699aec419775'), 'run_at': datetime.datetime(2026, 8, 8, 16, 8, 35, 992000), 'pages_craw

Testing the query interface

In [70]:
run_search_interface()

Softwarica Courses — Vertical Search Engine
Type 'exit' to quit.


Searching for: "bachelor computer science"

1. [0.1300] Programs & Courses | Softwarica College
   https://softwarica.edu.np/courses

2. [0.1054] BSc (Hons) Computer Science with Artificial Intelligence - Softwarica College
   https://softwarica.edu.np/courses/bsc-hons-computer-science-with-artificial-intelligence

3. [0.1054] BSc (Hons) Computer Science with Artificial Intelligence - Softwarica College
   https://softwarica.edu.np/course/computer-science-with-artificial-intelligence


Searching for: "ethical hacking"

1. [0.0877] BSc (Hons) Ethical Hacking and Cybersecurity - Softwarica College
   https://softwarica.edu.np/courses/bsc-hons-ethical-hacking-and-cybersecurity

2. [0.0816] Ethical Hacking Course in Nepal: Complete Guide- Colleges, Fees, Skills & Jobs | Softwarica College Blog
   https://softwarica.edu.np/student-center/blogs/ethical-hacking-course-in-nepal-complete-guide-colleges-fees-skills-jobs

3. [0.05

# Part C - Precision 5 evaluation

In [71]:
queries = [
    "bachelor computer science",
    "ethical hacking cybersecurity",
    "scholarship fees",
    "data science masters",
    "quantum blockchain nutrition",
]

judged = {}   # you'll fill this in Step 3

for q in queries:
    print(f"\n=== {q} ===")
    results = search(q, top_k=10)
    for rank, (score, url, title) in enumerate(results, start=1):
        print(f"{rank}. [{score:.4f}] {title}\n   {url}")


=== bachelor computer science ===
1. [0.1300] Programs & Courses | Softwarica College
   https://softwarica.edu.np/courses
2. [0.1054] BSc (Hons) Computer Science with Artificial Intelligence - Softwarica College
   https://softwarica.edu.np/courses/bsc-hons-computer-science-with-artificial-intelligence
3. [0.1054] BSc (Hons) Computer Science with Artificial Intelligence - Softwarica College
   https://softwarica.edu.np/course/computer-science-with-artificial-intelligence
4. [0.0696] Apply Now | Softwarica College
   https://softwarica.edu.np/apply
5. [0.0696] Apply Now | Softwarica College
   https://softwarica.edu.np/apply?utm_source=chatgpt.com
6. [0.0605] BSc (Hons) Computer Science with Artificial Intelligence vs BCA in Nepal | Softwarica College Blog
   https://softwarica.edu.np/student-center/blogs/bsc-hons-computer-science-with-artificial-intelligence-vs-bca-in-nepal
7. [0.0583] BSc (Hons) Computer Science with Artificial Intelligence- What You'll Learn - Career & Scope | Soft

In [72]:
# Judging the sites manually
# Fill these in by hand after reviewing each result against the live site.
# Order matches the rank order printed in Step 2 — one entry per result, top 10.
judged = {
    "bachelor computer science":        [1, 1, 1, 0, 0, 0, 1, 1, 0, 0],
    "ethical hacking cybersecurity":    [1, 1, 1, 1, 1, 0, 1, 0, 0, 1],
    "scholarship fees":                 [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
    "data science masters":             [0, 1, 1, 1, 1, 1, 0, 0, 0, 0],
    "quantum blockchain nutrition":     [0, 0, 0, 0, 0],
}

In [73]:
def precision_at_5(judgments):
    top5 = judgments[:5]
    if not top5:
        return None  # no results to judge
    return sum(top5) / 5

for q, j in judged.items():
    p5 = precision_at_5(j)
    print(f"{q}: Precision@5 = {p5 if p5 is not None else 'n/a'}")

bachelor computer science: Precision@5 = 0.6
ethical hacking cybersecurity: Precision@5 = 1.0
scholarship fees: Precision@5 = 0.6
data science masters: Precision@5 = 0.8
quantum blockchain nutrition: Precision@5 = 0.0


# Add result snippets

In [84]:
import re

def get_snippet(url, q_vector, doc_vector, max_len=200):
    # pick the query term this document scores highest on
    shared = set(q_vector) & set(doc_vector)
    if not shared:
        return ""
    best_term = max(shared, key=lambda t: q_vector[t] * doc_vector[t])

    page = raw_pages.find_one({"url": url}, {"text": 1})
    if not page:
        return ""
    text = page["text"]

    # split into naive sentences and find one whose stemmed tokens contain best_term
    sentences = re.split(r'(?<=[.!?])\s+', text)
    for sentence in sentences:
        if stemmer.stem_words if False else None:  # placeholder, see below
            pass
    for sentence in sentences:
        tokens = [stemmer.stem(w) for w in re.sub(r"[^a-zA-Z0-9\s]", " ", sentence.lower()).split()]
        if best_term in tokens:
            return (sentence[:max_len] + "…") if len(sentence) > max_len else sentence
    return (text[:max_len] + "…") if len(text) > max_len else text


def search(query, top_k=3):
    q_vector = build_query_vector(query)
    if not q_vector:
        print("No matching terms found in the index.")
        return []

    results = []
    for doc in doc_vectors.find({}):
        score = cosine_similarity(q_vector, doc["vector"])
        if score > 0:
            snippet = get_snippet(doc["url"], q_vector, doc["vector"])
            results.append((score, doc["url"], doc.get("title", ""), snippet))

    results.sort(key=lambda x: x[0], reverse=True)
    return results[:top_k]

In [85]:
search('ethical hacking networking')

[(0.07688408896678527,
  'https://softwarica.edu.np/courses/bsc-hons-ethical-hacking-and-cybersecurity',
  'BSc (Hons) Ethical Hacking and Cybersecurity - Softwarica College',
  'Loading Bachelors Degree undergraduate BSc (Hons) Ethical Hacking and Cybersecurity In this Ethical Hacking and Cybersecurity course, students will have the opportunity to learn to identify and analys…'),
 (0.0723007687068148,
  'https://softwarica.edu.np/student-center/blogs/ethical-hacking-course-in-nepal-complete-guide-colleges-fees-skills-jobs',
  'Ethical Hacking Course in Nepal: Complete Guide- Colleges, Fees, Skills & Jobs | Softwarica College Blog',
  'Loading Back to Blogs Technology Ethical Hacking Course in Nepal: Complete Guide- Colleges, Fees, Skills & Jobs Ethical Hacking Course in Nepal: Complete 2026 guide covering top colleges and institute…'),
 (0.05017573446165081,
  'https://softwarica.edu.np/student-center/blogs/scope-of-ethical-hacking-degree-course-in-nepal-future-demand-skills-opportuni